# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadFaizan0023/FlyRank_ML_internship_repo/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: RandomForestsClassifier
Reason: Due to multi-class nature, to take different pattern trees of same data, and choose average.
Here 'decline_Score' range from (0-17)

**1.1: Import libraries**

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import pandas as pd
import os, getpass
import duckdb
from sklearn.ensemble import RandomForestClassifier

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouped by client: Because if i will take time split, there would be biasness in the output for specific clients. May perform good for some and may not for some. And maybe it memorizes the patterns for certain clients and may not perform well on test set later on.

**2.1: Read data**

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
data = pd.read_csv("data_for_baseline_action_score.csv") # This file can be generated from previous baseline_score notebook.

**2.2: Check rows and cols**

In [3]:
data.shape

(4171375, 47)

**2.3: Handle missing values**

In [4]:
data.columns

Index(['client_hash_id', 'content_hash_id', 'gsc_impressions_prev30d',
       'gsc_clicks_prev30d', 'gsc_avg_position_prev30d',
       'ga4_pageviews_prev30d', 'ga4_sessions_prev30d', 'ga4_users_prev30d',
       ''ga4_engaged_session_prev30', 'ga4_total_engagement_sec_prev30',
       'sessions_organic_prev30', 'sessions_direct_prev30',
       'sessions_referral_prev30', 'sessions_social_prev30',
       'sessions_paid_prev30', 'sessions_ai_prev30', 'scroll_events_prev30',
       'content_created_date', 'content_updated_date', 'cpc_prev30',
       'backlinks_prev30', 'ctr_prev30', 'engagement_rate_prev30',
       'scroll_events_rate_prev30', 'days_since_update_prev30',
       'gsc_impressions_last30d', 'gsc_clicks_last30d',
       'gsc_avg_position_last30d', 'ga4_pageviews_last30d',
       'ga4_sessions_last30d', 'ga4_users_last30d',
       ''ga4_engaged_session_last30', 'ga4_total_engagement_sec_last30',
       'sessions_organic_last30', 'sessions_direct_last30',
       'sessions_referr

In [5]:
data["cpc_prev30"].isnull().sum()

np.int64(39006)

In [6]:
data['backlinks_prev30'].isnull().sum()

np.int64(839490)

In [7]:
data['engagement_rate_prev30'].isnull().sum()

np.int64(29333)

In [8]:
data['scroll_events_rate_prev30'].isnull().sum()

np.int64(3501)

In [9]:
data["cpc_last30"].isnull().sum()

np.int64(39006)

In [10]:
data["backlinks_last30"].isnull().sum()

np.int64(839490)

In [11]:
data["engagement_rate_last30"].isnull().sum()

np.int64(32574)

In [12]:
data["scroll_events_rate_last30"].isnull().sum()

np.int64(1691)

In [13]:
data = data.dropna(subset=['cpc_prev30']).reset_index(drop=True)

In [14]:
data = data.dropna(subset=['backlinks_prev30']).reset_index(drop=True)

In [15]:
data = data.dropna(subset=['engagement_rate_prev30']).reset_index(drop=True)

In [16]:
data = data.dropna(subset=['scroll_events_rate_prev30']).reset_index(drop=True)

In [17]:
data = data.dropna(subset=['engagement_rate_last30']).reset_index(drop=True)

In [18]:
data = data.dropna(subset=['scroll_events_rate_last30']).reset_index(drop=True)

**check again**

In [19]:
data["cpc_prev30"].isnull().sum()

np.int64(0)

In [20]:
data['backlinks_prev30'].isnull().sum()

np.int64(0)

In [21]:
data['engagement_rate_prev30'].isnull().sum()

np.int64(0)

In [22]:
data['scroll_events_rate_prev30'].isnull().sum()

np.int64(0)

In [23]:
data["cpc_last30"].isnull().sum()

np.int64(0)

In [24]:
data["backlinks_last30"].isnull().sum()

np.int64(0)

In [25]:
data["engagement_rate_last30"].isnull().sum()

np.int64(0)

In [26]:
data["scroll_events_rate_last30"].isnull().sum()

np.int64(0)

**2.4: Train-Test split = > Client hold-out split**

In [27]:
# Set random seed for reproducibility
np.random.seed(42)

# Get unique clients
unique_clients = data['client_hash_id'].unique()

# Shuffle and split client IDs (80/20)
np.random.shuffle(unique_clients)
split_idx = int(len(unique_clients) * 0.8)
train_clients = unique_clients[:split_idx]
test_clients = unique_clients[split_idx:]

# Split the dataset based on client groups
train_data = data[data['client_hash_id'].isin(train_clients)].copy()
test_data = data[data['client_hash_id'].isin(test_clients)].copy()

print(f"Total unique clients: {len(unique_clients)}")
print(f"Training clients: {len(train_clients)} | Rows: {len(train_data)}")
print(f"Testing clients: {len(test_clients)} | Rows: {len(test_data)}")

# Verify that there is no client overlap
overlap = set(train_data['client_hash_id']).intersection(set(test_data['client_hash_id']))
print(f"Client overlap between train and test: {len(overlap)}")

Total unique clients: 27
Training clients: 21 | Rows: 3127466
Testing clients: 6 | Rows: 146461
Client overlap between train and test: 0


In [28]:
train_data.shape

(3127466, 47)

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**3.1: Train and Test X-y split**

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
X_train = train_data.drop(columns=['client_hash_id', 'content_hash_id', 'content_created_date', 'content_updated_date', 'decline_score'], axis=1)
y_train = train_data['decline_score']

In [30]:
X_test = test_data.drop(columns=['client_hash_id', 'content_hash_id', 'content_created_date', 'content_updated_date', 'decline_score'], axis=1)

In [31]:
y_test = test_data['decline_score']

**3.2: Model training**

In [32]:
model = RandomForestClassifier(n_estimators = 20,random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=20,
                       random_state=42)

**3.3: Model predictions and evaluation**

In [33]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)

In [34]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           3       0.00      0.00      0.00         2
           4       0.00      0.00      0.00        17
           5       0.51      0.15      0.23       528
           6       0.45      0.56      0.50      2444
           7       0.45      0.57      0.50      4380
           8       0.43      0.41      0.42      4165
           9       0.44      0.46      0.45      4576
          10       0.50      0.45      0.47      7599
          11       0.52      0.61      0.56     11618
          12       0.55      0.52      0.54     13127
          13       0.58      0.57      0.58     14309
          14       0.58      0.59      0.59     15054
          15       0.71      0.66      0.69     23393
          16       0.73      0.84      0.78     30413
          17       0.90      0.62      0.74     14836

    accuracy                           0.63    146461
   macro avg       0.49      0.47      0.47    146461
weighted avg       0.64   

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [35]:
f1 = f1_score(y_test, y_pred, average='macro')
auc = roc_auc_score(y_test, y_proba, multi_class='ovo', labels=model.classes_)

print(f"F1: {f1:.4f}")
print(f"ROC-AUC: {auc:.4f}")

F1: 0.4694
ROC-AUC: 0.9315


**3.4: Comparison table**

In [36]:
print("Actual (y_test) vs Predicted (y_pred):")
comparison_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
display(comparison_df.head(20))
display(comparison_df.tail(20))
print(f"Total rows: {len(comparison_df)}\n")

Actual (y_test) vs Predicted (y_pred):


,Actual,Predicted
1276,17,17
1277,17,17
1289,17,16
1292,17,16
1293,17,17
1294,17,17
1295,17,16
1296,17,16
1297,17,16
1313,17,16


,Actual,Predicted
3272631,5,6
3272708,4,5
3272831,4,6
3272832,4,6
3272836,4,5
3272837,4,6
3272838,4,5
3272839,4,7
3272932,4,7
3273113,4,7


Total rows: 146461



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

**4.1: Confusion matrix**

In [38]:
print(confusion_matrix(y_test, y_pred))

[[    0     0     2     0     0     0     0     0     0     0     0     0
      0     0     0]
 [    0     0     3     3    11     0     0     0     0     0     0     0
      0     0     0]
 [    0     0    78   337   102     8     3     0     0     0     0     0
      0     0     0]
 [    0     0    34  1373   918    90    20     8     1     0     0     0
      0     0     0]
 [    0     0    28  1075  2512   634    94    33     3     1     0     0
      0     0     0]
 [    0     0     3   175  1342  1699   777   136    30     1     2     0
      0     0     0]
 [    0     0     1    47   326  1008  2109   860   200    22     3     0
      0     0     0]
 [    0     0     2    29   160   288  1199  3421  2210   247    37     6
      0     0     0]
 [    0     0     0    21   168   140   373  1710  7101  1855   222    24
      3     1     0]
 [    0     0     2     8    82    57   114   490  3239  6888  1999   229
     18     1     0]
 [    0     0     0     1    16    27    26   108 

**4.2: Classification report**

In [39]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           3       0.00      0.00      0.00         2
           4       0.00      0.00      0.00        17
           5       0.51      0.15      0.23       528
           6       0.45      0.56      0.50      2444
           7       0.45      0.57      0.50      4380
           8       0.43      0.41      0.42      4165
           9       0.44      0.46      0.45      4576
          10       0.50      0.45      0.47      7599
          11       0.52      0.61      0.56     11618
          12       0.55      0.52      0.54     13127
          13       0.58      0.57      0.58     14309
          14       0.58      0.59      0.59     15054
          15       0.71      0.66      0.69     23393
          16       0.73      0.84      0.78     30413
          17       0.90      0.62      0.74     14836

    accuracy                           0.63    146461
   macro avg       0.49      0.47      0.47    146461
weighted avg       0.64   

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


**Error analysis**

**Where is the model wrong**

The model's errors fall into two distinct groups with different causes:

**Tail classes (2, 3, 4)**: near-zero support (4, 138, and 1,379 examples respectively) means the model essentially never sees these during training and can't learn a boundary for them. This is a data volume limitation, not a modeling failure.
**Middle classes (8-13)**: despite having substantial support (64K-573K examples each), these score the weakest F1 (0.42-0.58) of any well-populated class range. Class 5 is a clear casualty of this same effect: reasonable support (8,927) but very low recall (0.15), meaning the model rarely predicts it even when it should, most consistent with adjacent classes absorbing it, matching the off-by-one confusion pattern found earlier in the confusion matrix.<br>
**What it leans on**

The model performs best not simply where data is most abundant, but where the feature signal is most internally consistent. Class 17, where all 17 decline conditions are triggered simultaneously, has only moderate support (212,004, ranked 8th of 16 by volume) yet achieves the best precision (0.90) and among the best F1 (0.74), outperforming several classes with 2-3x more training data. This works because every class-17 row shares the same extreme, unambiguous pattern (every signal declining at once), giving the model a clean, separable boundary regardless of row count.

The middle classes lack that consistency: many different combinations of which specific conditions fired can sum to the same decline_score, so two rows sharing a score like 10 or 11 may look quite different feature-wise. That inconsistency, not lack of data, is the likely driver of weak performance in this range.

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.